# Southend missed penalties

Every penalty Southend have missed, as far back as Wyscout has event data for them.

Wyscout lists Southend fixtures back to 2006/07, but the match events (which hold the penalties)
start in 2015/16. The coverage table below shows, for each season and competition, how many
played matches have events.

The data comes from `wyscout.data.penalties`. A penalty counts as missed when it did not go in:
saved, off target or hit the post. Shoot-out kicks are left out. A retaken penalty counts as two
kicks. "rebound_scored_by" shows a Southend goal within 10 seconds of the miss.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # CLIENT_ID and CLIENT_SECRET for the Wyscout API, from notebooks/.env
os.environ.setdefault("MONGO_HOST", "localhost")  # every API response is cached in Mongo

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)

from wyscout.data.penalties import team_penalties
from wyscout.ids import SOUTHEND

## Every penalty taken

The first run fetches events for every match and takes a while. After that it reads from the Mongo cache.

In [3]:
pens, coverage = team_penalties(SOUTHEND)

## How far back the data goes

In [4]:
coverage[coverage["with_events"] > 0]

,season,competition,played,with_events
17,2014/2015,League Two,49,3
20,2015/2016,League One,46,46
24,2016/2017,League One,46,46
25,2016/2017,Carabao Cup,1,1
26,2016/2017,EFL Vertu Trophy,4,4
29,2017/2018,League One,46,46
30,2017/2018,Carabao Cup,1,1
31,2017/2018,EFL Vertu Trophy,4,4
33,2018/2019,FA Cup,3,2
34,2018/2019,League One,46,46


In [5]:
print("First match with events:", pens["date"].min() if not pens.empty else None)
coverage[coverage["with_events"] == 0]

First match with events: 2015-05-14


,season,competition,played,with_events
0,2006/2007,Championship,46,0
1,2007/2008,League One,48,0
2,2007/2008,FA Cup,5,0
3,2008/2009,League One,46,0
4,2008/2009,FA Cup,5,0
5,2009/2010,League One,46,0
6,2009/2010,FA Cup,1,0
7,2010/2011,League Two,46,0
8,2011,Club Friendlies,2,0
9,2011/2012,League Two,48,0


## Missed penalties

In [6]:
missed = pens[pens["outcome"] != "Scored"].reset_index(drop=True)
missed[["date", "season", "competition", "opponent", "venue", "score", "taker", "minute", "outcome", "goalkeeper", "rebound_scored_by"]]

,date,season,competition,opponent,venue,score,taker,minute,outcome,goalkeeper,rebound_scored_by
0,2015-05-14,2014/2015,League Two,Stevenage,H,1-1,B. Corr,97,Hit post,NaN,NaN
1,2016-02-26,2015/2016,League One,Port Vale,A,1-3,D. Mooney,96,Hit post,NaN,NaN
2,2016-03-01,2015/2016,League One,Crewe Alexandra,H,1-1,T. Barnett,41,Saved,B. Garratt,NaN
3,2016-10-01,2016/2017,League One,Peterborough United,H,1-1,A. Wordsworth,64,Saved,L. McGee,NaN
4,2017-09-02,2017/2018,League One,Rochdale,H,0-0,S. Cox,48,Saved,B. Moore,NaN
5,2017-11-07,2017/2018,EFL Vertu Trophy,Colchester United,H,2-0,S. Cox,89,Saved,R. Gilmartin,NaN
6,2017-11-18,2017/2018,League One,Portsmouth,A,0-1,N. Ranger,46,Off target,NaN,NaN
7,2017-11-21,2017/2018,League One,Milton Keynes Dons,A,1-1,J. Wright,4,Saved,L. Nicholls,NaN
8,2020-09-08,2020/2021,EFL Vertu Trophy,West Ham United U21,H,1-3,S. Humphrys,35,Saved,J. Anang,NaN
9,2021-04-03,2020/2021,League Two,Carlisle United,H,0-2,S. Akinola,37,Saved,P. Farman,NaN


## Summary

In [7]:
print(f"Penalties taken: {len(pens)}, scored: {(pens['outcome'] == 'Scored').sum()}, missed: {len(missed)}")
pens.groupby("season")["outcome"].value_counts().unstack(fill_value=0)

Penalties taken: 65, scored: 46, missed: 19


outcome,Hit post,Off target,Saved,Scored
season,,,,
2014/2015,1,0,0,0
2015/2016,1,0,1,5
2016/2017,0,0,1,5
2017/2018,0,1,3,4
2018/2019,0,0,0,3
2019/2020,0,0,0,4
2020/2021,0,0,3,2
2021/2022,0,0,3,5
2022/2023,0,0,2,8


In [8]:
by_taker = pens.assign(scored=pens["outcome"] == "Scored").groupby("taker").agg(taken=("scored", "size"), scored=("scored", "sum"))
by_taker["missed"] = by_taker["taken"] - by_taker["scored"]
by_taker.sort_values(["missed", "taken"], ascending=False)

,taken,scored,missed
taker,,,
J. Bridge,15,13,2
S. Cox,7,5,2
C. Powell,2,0,2
J. Demetriou,9,8,1
D. Mooney,4,3,1
S. Dalby,4,3,1
A. Wordsworth,3,2,1
H. Cardwell,3,2,1
N. Ranger,2,1,1
